## Artificial Bee Colony

## Object Oriented Code

Convert to class using object oriented design

In [2]:
import numpy as np

class ArtificialBeeColony:
    def __init__(self, objective_function, num_food_sources=20, num_dimensions=2, max_iterations=100, limit=10, bounds=(-5, 5)):
        """
        Initialize the ABC algorithm.

        Parameters:
            objective_function (callable): The objective function to minimize.
            num_food_sources (int): Number of food sources (candidate solutions).
            num_dimensions (int): Number of dimensions in the search space.
            max_iterations (int): Maximum number of iterations.
            limit (int): Abandonment limit for scout bees.
            bounds (tuple): Lower and upper bounds for the search space.
        """
        self.objective_function = objective_function
        self.num_food_sources = num_food_sources
        self.num_dimensions = num_dimensions
        self.max_iterations = max_iterations
        self.limit = limit
        self.bounds = bounds

        # Initialize food sources (candidate solutions)
        self.food_sources = np.random.uniform(self.bounds[0], self.bounds[1], (self.num_food_sources, self.num_dimensions))
        self.fitness_values = np.zeros(self.num_food_sources)
        self.trials = np.zeros(self.num_food_sources)
        self.best_solution = None
        self.best_fitness = float('inf')

    def fitness(self, f):
        """
        Calculate the fitness of a solution.

        Parameters:
            f (float): Objective function value.

        Returns:
            float: Fitness value.
        """
        return 1 / (1 + f) if f >= 0 else 1 + abs(f)

    def evaluate_fitness(self):
        """
        Evaluate the fitness of all food sources.
        """
        for i in range(self.num_food_sources):
            f = self.objective_function(self.food_sources[i])
            self.fitness_values[i] = self.fitness(f)
            if f < self.best_fitness:
                self.best_fitness = f
                self.best_solution = self.food_sources[i]

    def employed_bee_phase(self):
        """
        Employed bee phase: Explore new solutions around existing food sources.
        """
        for i in range(self.num_food_sources):
            # Select a random neighbor
            rand_neighbor = np.random.choice([j for j in range(self.num_food_sources) if j != i])
            # Select a random dimension
            rand_dim = np.random.randint(self.num_dimensions)

            # Generate a new solution
            solution = np.copy(self.food_sources[i])
            phi = np.random.uniform(-1, 1)
            # 𝑋_𝑛𝑒𝑤 = 𝑋 + 𝜙 * (𝑋 − 𝑋_𝑝 )
            solution[rand_dim] = self.food_sources[i][rand_dim] + phi * (self.food_sources[i][rand_dim] - self.food_sources[rand_neighbor][rand_dim])

            # Clip the solution to stay within bounds
            solution = np.clip(solution, self.bounds[0], self.bounds[1])
           
            # Evaluate the new solution
            # calculate the objective function value for the new solution
            f_x = self.objective_function(solution)
            # calculate the fitness for the new solution
            new_fitness = self.fitness(f_x)
            if new_fitness > self.fitness_values[i]:  # If the new solution is better
                self.food_sources[i] = solution
                self.fitness_values[i] = new_fitness
                self.trials[i] = 0  # Reset trials for this food source
            else:
                self.trials[i] += 1  # Increment trials for this food source
            

    def onlooker_bee_phase(self):
        """
        Onlooker bee phase: Select food sources based on fitness and explore around them.
        """
        probabilities = self.fitness_values / np.sum(self.fitness_values)
        for i in range(self.num_food_sources):
            if np.random.rand() < probabilities[i]:
                # Perform the same steps as the employed bee phase
                self.employed_bee_phase()

    def scout_bee_phase(self):
        """
        Scout bee phase: Replace abandoned food sources with new random solutions.
        """
        for i in range(self.num_food_sources):
            if self.trials[i] > self.limit:
                self.food_sources[i] = np.random.uniform(self.bounds[0], self.bounds[1], self.num_dimensions)
                self.trials[i] = 0  # Reset trials for this food source

    def optimize(self):
        """
        Run the ABC optimization algorithm.

        Returns:
            tuple: Best solution and best fitness value.
        """
        self.evaluate_fitness()  # Evaluate initial fitness

        for iteration in range(self.max_iterations):
            self.employed_bee_phase()
            self.onlooker_bee_phase()
            self.scout_bee_phase()
            self.evaluate_fitness()  # Update best solution

            print(f"Iteration {iteration + 1}: Best Fitness = {self.best_fitness}")

        return self.best_solution, self.best_fitness

In [3]:
# Define the objective function (e.g., sphere function)
def sphere_function(x):
    return sum(x**2)

# Initialize and run the ABC algorithm
abc = ArtificialBeeColony(objective_function=sphere_function, num_food_sources=20, num_dimensions=2, max_iterations=10)
best_solution, best_fitness = abc.optimize()

print("\nBest Solution:", best_solution)
print("Best Fitness:", best_fitness)

Iteration 1: Best Fitness = 1.070378290418825
Iteration 2: Best Fitness = 0.0006866120424971168
Iteration 3: Best Fitness = 0.0006866120424971168
Iteration 4: Best Fitness = 0.0006866120424971168
Iteration 5: Best Fitness = 0.0006866120424971168
Iteration 6: Best Fitness = 0.0006866120424971168
Iteration 7: Best Fitness = 0.0006866120424971168
Iteration 8: Best Fitness = 0.0006866120424971168
Iteration 9: Best Fitness = 0.0006866120424971168
Iteration 10: Best Fitness = 0.0006866120424971168

Best Solution: [ 1.94282472 -0.72795233]
Best Fitness: 0.0006866120424971168


In [4]:
import numpy as np
import pygame
import sys

class ABCAnimation(ArtificialBeeColony):
    def __init__(self, objective_function, num_food_sources=20, num_dimensions=2, max_iterations=100, limit=10, bounds=(-5, 5)):
        """
        Initialize the ABC algorithm.

        Parameters:
            objective_function (callable): The objective function to minimize.
            num_food_sources (int): Number of food sources (candidate solutions).
            num_dimensions (int): Number of dimensions in the search space.
            max_iterations (int): Maximum number of iterations.
            limit (int): Abandonment limit for scout bees.
            bounds (tuple): Lower and upper bounds for the search space.
        """     
        super().__init__(objective_function, num_food_sources, num_dimensions, max_iterations, limit, bounds)

    def optimize(self, screen=None, scale=None):
        """
        Run the ABC optimization algorithm.

        Returns:
            tuple: Best solution and best fitness value.
        """
        self.evaluate_fitness()  # Evaluate initial fitness

        for iteration in range(self.max_iterations):
            self.employed_bee_phase()
            self.onlooker_bee_phase()
            self.scout_bee_phase()
            self.evaluate_fitness()  # Update best solution

            print(f"Iteration {iteration + 1}: Best Solution = {self.best_solution}, Best Fitness = {self.best_fitness}")

            if screen and scale:
                self.draw(screen, scale)
                pygame.display.flip()
                pygame.time.delay(50)  # Adjust delay for animation speed

        return self.best_solution, self.best_fitness
   
    def draw(self, screen, scale):
        """Draw the bees on the screen."""
        
        screen.fill((255, 255, 255))  # White background
        for bee in self.food_sources:
            x = int((bee[0] - self.bounds[0]) * scale)
            y = int((bee[1] - self.bounds[0]) * scale)
            pygame.draw.circle(screen, (0, 0, 255), (x, y), 5)  # Blue bees
            
        if self.best_solution is not None:
            x_best = int((self.best_solution[0] - self.bounds[0]) * scale)
            y_best = int((self.best_solution[1] - self.bounds[0]) * scale)
            pygame.draw.circle(screen, (255, 0, 0), (x_best, y_best), 7)  # Red best bee

In [5]:
def sphere_function(x):
    return sum(x**2)

pygame.init()
screen_size = 600
screen = pygame.display.set_mode((screen_size, screen_size))
pygame.display.set_caption("ABC Optimization")

abc = ABCAnimation(objective_function=sphere_function, num_food_sources=20, num_dimensions=2, max_iterations=100)
best_solution, best_fitness = abc.optimize(screen, screen_size / (abc.bounds[1] - abc.bounds[0]))

print("\nBest Solution:", best_solution)
print("Best Fitness:", best_fitness)

#wait for close
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

Iteration 1: Best Solution = [-1.0928013  0.3901221], Best Fitness = 1.3464099420761273
Iteration 2: Best Solution = [0.90251589 0.49592093], Best Fitness = 1.060472512482246
Iteration 3: Best Solution = [0.90251589 0.49592093], Best Fitness = 1.060472512482246
Iteration 4: Best Solution = [-0.13512045  0.23530705], Best Fitness = 0.07362694543941266
Iteration 5: Best Solution = [-0.13512045  0.23530705], Best Fitness = 0.07362694543941266
Iteration 6: Best Solution = [-0.13512045  0.23530705], Best Fitness = 0.07362694543941266
Iteration 7: Best Solution = [-0.05015909  0.26228049], Best Fitness = 0.07130698791133623
Iteration 8: Best Solution = [-0.05015909  0.08141896], Best Fitness = 0.00914498094353065
Iteration 9: Best Solution = [-0.00685039 -0.03176954], Best Fitness = 0.0010562315916171174
Iteration 10: Best Solution = [-0.00685039 -0.03176954], Best Fitness = 0.0010562315916171174
Iteration 11: Best Solution = [-0.00685039 -0.03176954], Best Fitness = 0.0010562315916171174
It

SystemExit: 

c:\Users\Shira\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Test with other functions for 2 Dimensions

1. Rastrigin (https://www.sfu.ca/~ssurjano/rastr.html)
2. Styblinski-Tang (https://www.sfu.ca/~ssurjano/stybtang.html)
3. Eggholder (https://www.sfu.ca/~ssurjano/egg.html)

General functions available at: https://www.sfu.ca/~ssurjano/optimization.html

In [6]:
def rastrigin_function(x):
    A = 10
    return A * len(x) + sum([(xi - 1)**2 - A * np.cos(2 * np.pi * (xi - 1)) for xi in x])

def styblunski_tang_function(x):
    return 0.5 * sum([xi**4 - 16*xi**2 + 5*xi for xi in x])

def eggholder_function(x):
    x1, x2 = x
    return -(x2 + 47) * np.sin(np.sqrt(abs(x2 + x1/2 + 47))) - x1 * np.sin(np.sqrt(abs(x1 - x2 - 47)))

In [7]:
pygame.init()
screen_size = 600
screen = pygame.display.set_mode((screen_size, screen_size))
pygame.display.set_caption("ABC Optimization")

abc = ABCAnimation(objective_function=rastrigin_function, num_food_sources=20, num_dimensions=2, max_iterations=100, bounds=(-5.12, 5.12))
best_solution, best_fitness = abc.optimize(screen, screen_size / (abc.bounds[1] - abc.bounds[0]))

print("\nBest Solution:", best_solution)
print("Best Fitness:", best_fitness)

#wait for close
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

Iteration 1: Best Solution = [-1.01665637  1.89986226], Best Fitness = 6.8462884042586545
Iteration 2: Best Solution = [1.95841178 2.94367497], Best Fitness = 5.655612167419445
Iteration 3: Best Solution = [2.03137336 2.94367497], Best Fitness = 5.654985031698043
Iteration 4: Best Solution = [2.03137336 2.94367497], Best Fitness = 5.654985031698043
Iteration 5: Best Solution = [2.03137336 2.94367497], Best Fitness = 5.654985031698043
Iteration 6: Best Solution = [2.03137336 2.94367497], Best Fitness = 5.654985031698043
Iteration 7: Best Solution = [1.06385013 2.94853775], Best Fitness = 5.1131035315538185
Iteration 8: Best Solution = [ 0.94087561 -1.00361718], Best Fitness = 4.702683275814973
Iteration 9: Best Solution = [1.02406871 3.00921785], Best Fitness = 4.16843515555416
Iteration 10: Best Solution = [1.02406871 3.00921785], Best Fitness = 4.16843515555416
Iteration 11: Best Solution = [ 1.0408447  -0.00531671], Best Fitness = 1.3454129864266307
Iteration 12: Best Solution = [ 1.

SystemExit: 

In [ ]:
pygame.init()
screen_size = 600
screen = pygame.display.set_mode((screen_size, screen_size))
pygame.display.set_caption("ABC Optimization")

abc = ABCAnimation(objective_function=styblunski_tang_function, num_food_sources=20, num_dimensions=2, max_iterations=100, bounds=(-5, 5))
best_solution, best_fitness = abc.optimize(screen, screen_size / (abc.bounds[1] - abc.bounds[0]))

print("\nBest Solution:", best_solution)
print("Best Fitness:", best_fitness)

#wait for close
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

Iteration 1: Best Fitness = -63.51160042365677
Iteration 2: Best Fitness = -65.64941678167638
Iteration 3: Best Fitness = -65.64941678167638
Iteration 4: Best Fitness = -78.2093436421085
Iteration 5: Best Fitness = -78.25416461944309
Iteration 6: Best Fitness = -78.25416461944309
Iteration 7: Best Fitness = -78.25416461944309
Iteration 8: Best Fitness = -78.32124266831056
Iteration 9: Best Fitness = -78.32124266831056
Iteration 10: Best Fitness = -78.32124266831056
Iteration 11: Best Fitness = -78.32124266831056
Iteration 12: Best Fitness = -78.32124266831056
Iteration 13: Best Fitness = -78.32124266831056
Iteration 14: Best Fitness = -78.32124266831056
Iteration 15: Best Fitness = -78.33104015668343
Iteration 16: Best Fitness = -78.33104015668343
Iteration 17: Best Fitness = -78.33104015668343
Iteration 18: Best Fitness = -78.33106620167345
Iteration 19: Best Fitness = -78.33106620167345
Iteration 20: Best Fitness = -78.33106620167345
Iteration 21: Best Fitness = -78.33229314208522
It

SystemExit: 

In [8]:
pygame.init()
screen_size = 600
screen = pygame.display.set_mode((screen_size, screen_size))
pygame.display.set_caption("ABC Optimization")

abc = ABCAnimation(objective_function=eggholder_function, num_food_sources=20, num_dimensions=2, max_iterations=100, bounds=(-512, 512))
best_solution, best_fitness = abc.optimize(screen, screen_size / (abc.bounds[1] - abc.bounds[0]))

print("\nBest Solution:", best_solution)
print("Best Fitness:", best_fitness)

#wait for close
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

Iteration 1: Best Solution = [361.92350901 512.        ], Best Fitness = -851.0759506874447
Iteration 2: Best Solution = [361.92350901 512.        ], Best Fitness = -851.0759506874447
Iteration 3: Best Solution = [361.92350901 512.        ], Best Fitness = -851.0759506874447
Iteration 4: Best Solution = [361.92350901 512.        ], Best Fitness = -851.0759506874447
Iteration 5: Best Solution = [-311.04724233  162.57243833], Best Fitness = -851.0759506874447
Iteration 6: Best Solution = [-311.04724233  162.57243833], Best Fitness = -851.0759506874447
Iteration 7: Best Solution = [-311.04724233  162.57243833], Best Fitness = -851.0759506874447
Iteration 8: Best Solution = [-311.04724233  162.57243833], Best Fitness = -851.0759506874447
Iteration 9: Best Solution = [-311.04724233  162.57243833], Best Fitness = -851.0759506874447
Iteration 10: Best Solution = [-311.04724233  162.57243833], Best Fitness = -851.0759506874447
Iteration 11: Best Solution = [-311.04724233  162.57243833], Best F

SystemExit: 